# Bölüm 2: Veriyi Özetleme
**Haydar Kılıç — Yapay Zeka Mühendisliği**

Bu notebook, ders sunumundaki tüm teorik kavramların Python uygulamalarını içermektedir.

---
## İçindekiler
1. [Sayısal Verilerin İncelenmesi](#1)
   - Saçılım Grafiği
   - Nokta Grafikleri
   - Histogramlar & Aralık Genişliği
   - Dağılım Şekli (Tepe Sayısı & Çarpıklık)
2. [Merkez ve Yayılım Ölçüleri](#2)
   - Ortalama & Varyans & Standart Sapma
   - Ortanca, Q1, Q3, IQR
   - Kutu Grafiği
   - Sağlam İstatistikler
3. [Logaritmik Dönüşüm](#3)
4. [Kategorik Verilerin İncelenmesi](#4)
   - Çapraz Tablolar
   - Çubuk Grafikler
   - Mozaik Grafikleri
5. [Örnek Çalışma: Cinsiyete Dayalı Ayrımcılık & Hipotez Testi](#5)

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Renk paleti
BLUE   = '#4C88C4'
LBLUE  = '#A8C8E8'
ORANGE = '#E8834C'
GREEN  = '#5BAD72'
GRAY   = '#888888'

np.random.seed(42)
print('Kütüphaneler başarıyla yüklendi ✓')

---
<a id='1'></a>
## 1. Sayısal Verilerin İncelenmesi

### 1.1 Saçılım Grafiği (Scatter Plot)

> **Tanım:** Saçılım grafikleri, **iki sayısal değişken arasındaki ilişkiyi** görselleştirmek için kullanılır.

Aşağıdaki örnek, ders sunumundaki **yaşam beklentisi – toplam doğurganlık** ilişkisini simüle etmektedir.

In [ ]:
# Gapminder benzeri veri simülasyonu
n = 150
# Doğurganlık: 1.1 ile 7.5 arasında
fertility = np.random.uniform(1.1, 7.5, n)
# Yaşam beklentisi: doğurganlıkla negatif doğrusal ilişki + gürültü
life_exp  = 85 - 6.5 * fertility + np.random.normal(0, 4, n)
life_exp  = np.clip(life_exp, 30, 85)
# Nüfus (baloncuk boyutu için)
population = np.random.exponential(50, n) + 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, alpha, title in zip(axes, [0.6, 0.4], ['2000', '2020']):
    scatter = ax.scatter(
        fertility, life_exp,
        s=population * 3,
        c=np.random.choice([BLUE, ORANGE, GREEN, '#C45CA0'], n),
        alpha=alpha, edgecolors='white', linewidths=0.5
    )
    # Trend çizgisi
    z = np.polyfit(fertility, life_exp, 1)
    p = np.poly1d(z)
    x_line = np.linspace(fertility.min(), fertility.max(), 100)
    ax.plot(x_line, p(x_line), color=ORANGE, linewidth=2, linestyle='--', alpha=0.8, label='Trend')
    ax.set_xlabel('Kadın Başına Çocuk (Toplam Doğurganlık)', fontsize=10)
    ax.set_ylabel('Yaşam Beklentisi (yıl)', fontsize=10)
    ax.set_title(f'Yaşam Beklentisi vs Doğurganlık — {title}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Saçılım Grafiği Örneği (Gapminder benzeri)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Korelasyon katsayısı
r, p_val = stats.pearsonr(fertility, life_exp)
print(f'Pearson Korelasyon Katsayısı (r) = {r:.3f}')
print(f'Yorum: Doğurganlık arttıkça yaşam beklentisi {'azalıyor' if r < 0 else 'artıyor'} '
      f'(negatif doğrusal ilişki).')

### 1.2 Nokta Grafiği ve Ortalama

> **Tanım:** Tek bir sayısal değişkeni görselleştirmek için kullanılır. Koyu renkler daha fazla gözlemin bulunduğu alanları temsil eder.

**Örneklem ortalaması:**
$$\bar{x} = \frac{x_1 + x_2 + \cdots + x_n}{n}$$

In [ ]:
# Not ortalaması (GPA) verisi simülasyonu
# Sunumdaki grafikteki dağılımı taklit ediyoruz: sola hafif çarpık, 3.3-4.0 arası yoğun
gpa_data = np.concatenate([
    np.random.normal(3.6, 0.3, 200),
    np.random.uniform(2.5, 3.0, 30)
])
gpa_data = np.clip(gpa_data, 2.5, 4.0)

mean_gpa = np.mean(gpa_data)
median_gpa = np.median(gpa_data)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# --- Nokta grafiği (jitter ile) ---
ax = axes[0]
y_jitter = np.random.uniform(-0.15, 0.15, len(gpa_data))
ax.scatter(gpa_data, y_jitter, alpha=0.35, s=18, color=BLUE, edgecolors='none')
ax.axvline(mean_gpa, color=ORANGE, linewidth=2, label=f'Ortalama = {mean_gpa:.2f}')
ax.plot(mean_gpa, 0, marker='^', color=ORANGE, markersize=14, zorder=5)
ax.set_yticks([])
ax.set_xlabel('GPA', fontsize=11)
ax.set_xlim(2.4, 4.05)
ax.set_title('Nokta Grafiği — GPA Dağılımı', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# --- Yığılmış nokta grafiği (histogram şeklinde) ---
ax2 = axes[1]
ax2.hist(gpa_data, bins=30, color=BLUE, alpha=0.7, edgecolor='white')
ax2.axvline(mean_gpa,   color=ORANGE, linewidth=2, linestyle='-',  label=f'Ortalama  = {mean_gpa:.2f}')
ax2.axvline(median_gpa, color=GREEN,  linewidth=2, linestyle='--', label=f'Ortanca = {median_gpa:.2f}')
ax2.set_xlabel('GPA', fontsize=11)
ax2.set_ylabel('Frekans', fontsize=11)
ax2.set_title('Yığılmış Nokta Grafiği (Histogram) — GPA Dağılımı', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'n                = {len(gpa_data)}')
print(f'Örneklem Ortalaması (x̄) = {mean_gpa:.3f}')
print(f'Ortanca          = {median_gpa:.3f}')

### 1.3 Histogramlar ve Aralık Genişliği

> **Önemli:** Seçilen **aralık genişliği (bin width)** histogramın anlattığı hikayeyi değiştirebilir.

Çok geniş aralıklar → fazla bilgi gizlenir  
Çok dar aralıklar → fazla gürültü görünür

In [ ]:
# Ders dışı faaliyetler verisi (sağa çarpık)
extracurricular = np.concatenate([
    np.random.exponential(8, 150),
    np.random.uniform(0, 5, 50),
    [60, 65, 70]  # aykırı değerler
])
extracurricular = np.clip(extracurricular, 0, 75)

bin_configs = [
    (5,  'Çok Geniş Aralık\n(çok fazlası gizleniyor)'),
    (10, 'İyi Aralık\n(dengeli)'),
    (20, 'Orta Aralık'),
    (50, 'Çok Dar Aralık\n(çok fazla ayrıntı / gürültü)'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (bins, title) in zip(axes, bin_configs):
    ax.hist(extracurricular, bins=bins, color=BLUE, alpha=0.75, edgecolor='white')
    ax.set_xlabel('Haftada Harcanan Saat (Ders Dışı Faaliyetler)', fontsize=9)
    ax.set_ylabel('Frekans', fontsize=9)
    ax.set_title(f'{title}  (bins={bins})', fontsize=10, fontweight='bold')

plt.suptitle('Aralık Genişliğinin Histogram Üzerindeki Etkisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.4 Dağılımın Şekli: Tepe Sayısı ve Çarpıklık

| Özellik | Terimler |
|---------|----------|
| Tepe sayısı | Tek tepeli, İki tepeli, Çok tepeli, Düzgün |
| Çarpıklık | Sağa çarpık, Sola çarpık, Simetrik |

> **Kural:** Histogramlar, **uzun kuyruk tarafına** doğru çarpık olduğu söylenir.

In [ ]:
# 6 farklı dağılım şekli
dists = {
    'Tek Tepeli\n(Simetrik)':    np.random.normal(10, 2, 1000),
    'İki Tepeli':                np.concatenate([np.random.normal(5, 1, 500),
                                                  np.random.normal(15, 1, 500)]),
    'Çok Tepeli':                np.concatenate([np.random.normal(3, 0.8, 300),
                                                  np.random.normal(8, 0.8, 300),
                                                  np.random.normal(14, 0.8, 400)]),
    'Düzgün Dağılım':            np.random.uniform(0, 20, 1000),
    'Sağa Çarpık':               np.random.exponential(3, 1000),
    'Sola Çarpık':               20 - np.random.exponential(3, 1000),
}

colors = [BLUE, GREEN, ORANGE, '#9B59B6', '#E74C3C', '#1ABC9C']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, (title, data), color in zip(axes, dists.items(), colors):
    ax.hist(data, bins=35, color=color, alpha=0.8, edgecolor='white')
    mean_val   = np.mean(data)
    median_val = np.median(data)
    ax.axvline(mean_val,   color='navy',  linewidth=2, linestyle='--', label=f'Ort. = {mean_val:.1f}')
    ax.axvline(median_val, color='black', linewidth=2, linestyle='-',  label=f'Med. = {median_val:.1f}')
    skewness = stats.skew(data)
    ax.set_title(f'{title}\n(çarpıklık = {skewness:.2f})', fontsize=10, fontweight='bold')
    ax.set_yticks([])
    ax.legend(fontsize=8)

plt.suptitle('Dağılım Şekilleri: Tepe Sayısı ve Çarpıklık', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Çarpıklık Yorumu:')
print('  skewness > 0  → Sağa çarpık  (ortalama > ortanca)')
print('  skewness < 0  → Sola çarpık  (ortalama < ortanca)')
print('  skewness ≈ 0  → Simetrik     (ortalama ≈ ortanca)')

---
<a id='2'></a>
## 2. Merkez ve Yayılım Ölçüleri

### 2.1 Varyans ve Standart Sapma

**Varyans** — ortalamadan karesel sapmanın ortalaması:
$$s^2 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}$$

**Standart Sapma** — varyansın karekökü (verilerle aynı birimde):
$$s = \sqrt{s^2}$$

> **Neden $n-1$?** Serbestlik derecesi kaybı (Bessel düzeltmesi) — örneklem istatistiğinin kitle varyansını yansız tahmin etmesi için.

In [ ]:
# Sunumdaki uyku verisi
sleep_data = np.concatenate([
    np.random.normal(7, 1.5, 180),
    np.random.normal(4, 0.5, 20),
    np.random.normal(10, 0.5, 17)
])
sleep_data = np.clip(sleep_data, 2, 12)

# --- Manuel hesaplama ---
n       = len(sleep_data)
x_bar   = np.mean(sleep_data)
var_s2  = np.sum((sleep_data - x_bar)**2) / (n - 1)   # örneklem varyansı
std_s   = np.sqrt(var_s2)

# --- NumPy ile doğrulama ---
assert abs(var_s2 - np.var(sleep_data, ddof=1)) < 1e-10
assert abs(std_s  - np.std(sleep_data, ddof=1)) < 1e-10

# --- Görselleştirme ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sleep_data, bins=20, color=BLUE, alpha=0.75, edgecolor='white', label='Gözlemler')
ax.axvline(x_bar, color=ORANGE, linewidth=2.5, label=f'Ortalama (x̄) = {x_bar:.2f} saat')

# ±1s ve ±2s ve ±3s bantları
for k, alpha in [(1, 0.20), (2, 0.12), (3, 0.07)]:
    ax.axvspan(x_bar - k*std_s, x_bar + k*std_s, alpha=alpha, color=ORANGE,
               label=f'±{k}s  ({x_bar - k*std_s:.1f} – {x_bar + k*std_s:.1f})')

ax.set_xlabel('Gecelik Uyku Süresi (saat)', fontsize=11)
ax.set_ylabel('Frekans', fontsize=11)
ax.set_title('Uyku Süresi: Ortalama ve Standart Sapma', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

print('=' * 45)
print(f'Örneklem Büyüklüğü   n  = {n}')
print(f'Örneklem Ortalaması  x̄  = {x_bar:.2f} saat')
print(f'Örneklem Varyansı    s² = {var_s2:.2f} saat²')
print(f'Standart Sapma       s  = {std_s:.2f} saat')
print('=' * 45)

# Kaç veri noktası 3s içinde?
within_3s = np.sum(np.abs(sleep_data - x_bar) <= 3 * std_s)
print(f'\n3 standart sapma içindeki veri: {within_3s}/{n} = {within_3s/n*100:.1f}%')

### 2.2 Ortanca, Çeyrekler ve IQR

| Kavram | Tanım |
|--------|-------|
| **Ortanca (Median)** | Veriyi ikiye bölen değer — 50. yüzdelik dilim |
| **Q1** | 25. yüzdelik dilim (1. çeyrek) |
| **Q3** | 75. yüzdelik dilim (3. çeyrek) |
| **IQR** | Q3 − Q1 (orta %50'lik aralık) |

In [ ]:
# Küçük örnek — adım adım hesaplama
ornek = [4, 7, 13, 2, 1, 9, 6, 11, 5]
ornek_sorted = sorted(ornek)
print('Sıralanmış veri:', ornek_sorted)

median_val = np.median(ornek)
q1         = np.percentile(ornek, 25)
q3         = np.percentile(ornek, 75)
iqr        = q3 - q1

print(f'Ortanca (Median) = {median_val}')
print(f'Q1 (25. yüzd.)   = {q1}')
print(f'Q3 (75. yüzd.)   = {q3}')
print(f'IQR              = Q3 - Q1 = {q3} - {q1} = {iqr}')

# Çift sayıda örnek
ornek2 = [0, 1, 2, 3, 4, 5]
print(f'\nÇift sayıda örnek {ornek2}: Ortanca = ({ornek2[2]} + {ornek2[3]}) / 2 = {np.median(ornek2)}')

### 2.3 Kutu Grafiği (Box Plot)

> **Bıyık Kuralı:** Bıyıklar çeyreklerden **1.5 × IQR** uzaklığa kadar uzanır.
> - Üst bıyık üst sınırı = Q3 + 1.5 × IQR
> - Alt bıyık üst sınırı = Q1 − 1.5 × IQR
> - Bu sınırların ötesindeki noktalar **potansiyel aykırı değer** olarak işaretlenir.

In [ ]:
# Haftalık çalışma saati verisi (sağa çarpık + aykırı değerler)
study_hours = np.concatenate([
    np.random.normal(15, 6, 180),
    [45, 50, 55, 60, 65, 70]  # aykırı değerler
])
study_hours = np.clip(study_hours, 0, 75)

Q1  = np.percentile(study_hours, 25)
Q3  = np.percentile(study_hours, 75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# --- Açıklamalı kutu grafiği ---
ax = axes[0]
bp = ax.boxplot(study_hours, vert=True, patch_artist=True,
                boxprops=dict(facecolor=LBLUE, color=BLUE, linewidth=2),
                medianprops=dict(color='black', linewidth=2.5),
                whiskerprops=dict(color=BLUE, linewidth=1.5),
                capprops=dict(color=BLUE, linewidth=2),
                flierprops=dict(marker='o', color=ORANGE, alpha=0.7, markersize=6))

median_val = np.median(study_hours)

# Etiketler
offset = 0.55
for label, yval, color in [
    (f'Q1 = {Q1:.1f}',         Q1,          GREEN),
    (f'Ortanca = {median_val:.1f}', median_val, 'black'),
    (f'Q3 = {Q3:.1f}',         Q3,          ORANGE),
    (f'Alt Bıyık ≈ {max(alt_sinir,0):.1f}', max(alt_sinir, 0), BLUE),
    (f'Üst Bıyık ≈ {ust_sinir:.1f}', ust_sinir, BLUE),
]:
    ax.annotate(label, xy=(1, yval), xytext=(1 + offset, yval),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2),
                fontsize=9, color=color, va='center')

ax.set_ylabel('Haftalık Çalışma Saati', fontsize=11)
ax.set_title('Kutu Grafiğinin Anatomisi', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_xlim(0.5, 2.2)

# --- Kutu grafiği + veri noktaları ---
ax2 = axes[1]
jitter = np.random.uniform(-0.12, 0.12, len(study_hours))
outliers_mask = (study_hours > ust_sinir) | (study_hours < alt_sinir)
ax2.scatter(1 + jitter[~outliers_mask], study_hours[~outliers_mask],
            alpha=0.3, s=15, color=BLUE, label='Normal gözlemler')
ax2.scatter(1 + jitter[outliers_mask], study_hours[outliers_mask],
            alpha=0.8, s=40, color=ORANGE, label='Potansiyel aykırı değer', zorder=5)
ax2.boxplot(study_hours, vert=True, patch_artist=True,
            boxprops=dict(facecolor=LBLUE, color=BLUE, alpha=0.4, linewidth=1.5),
            medianprops=dict(color='black', linewidth=2),
            whiskerprops=dict(color=BLUE, linewidth=1.5),
            capprops=dict(color=BLUE, linewidth=2),
            showfliers=False)
ax2.set_ylabel('Haftalık Çalışma Saati', fontsize=11)
ax2.set_title('Kutu Grafiği + Veri Noktaları', fontsize=12, fontweight='bold')
ax2.set_xticks([])
ax2.legend(fontsize=9)

plt.suptitle('Kutu Grafiği (Box Plot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Q1  = {Q1:.2f}    Q3  = {Q3:.2f}    IQR = {IQR:.2f}')
print(f'Alt bıyık üst sınırı = Q1 - 1.5×IQR = {alt_sinir:.2f}')
print(f'Üst bıyık üst sınırı = Q3 + 1.5×IQR = {ust_sinir:.2f}')
print(f'Aykırı değer sayısı  = {outliers_mask.sum()}')

### 2.4 Sağlam İstatistikler

| İstatistik | Aykırı Değere Karşı | Önerilen Kullanım |
|------------|---------------------|--------------------|
| **Ortanca & IQR** | ✅ Sağlam | Çarpık dağılımlar |
| **Ortalama & s** | ❌ Hassas | Simetrik dağılımlar |

In [ ]:
# Hane geliri verisi (sunumdaki tablonun yeniden üretimi)
np.random.seed(7)
gelir = np.concatenate([
    np.random.lognormal(np.log(190_000), 0.5, 200),
    np.random.uniform(800_000, 1_000_000, 20)
])

def ozet_tablo(data, etiket):
    Q1  = np.percentile(data, 25)
    Q3  = np.percentile(data, 75)
    return {
        'Senaryo':    etiket,
        'Ortanca':    f'{np.median(data):>10,.0f} ₺',
        'IQR':        f'{Q3-Q1:>10,.0f} ₺',
        'Ortalama':   f'{np.mean(data):>10,.0f} ₺',
        'Std.Sapma':  f'{np.std(data, ddof=1):>10,.0f} ₺',
    }

# 3 senaryo
gelir_buyuk_degisti = gelir.copy(); gelir_buyuk_degisti[np.argmax(gelir)] = 10_000_000
gelir_kucuk_degisti = gelir.copy(); gelir_kucuk_degisti[np.argmin(gelir)] = 10_000_000

df_ozet = pd.DataFrame([
    ozet_tablo(gelir,               'Orijinal veri'),
    ozet_tablo(gelir_buyuk_degisti, 'En büyük → 10 milyon ₺'),
    ozet_tablo(gelir_kucuk_degisti, 'En küçük → 10 milyon ₺'),
])
df_ozet.set_index('Senaryo', inplace=True)

print('Sağlam İstatistikler Karşılaştırması')
print('=' * 72)
print(df_ozet.to_string())
print('=' * 72)
print('\n→ Ortanca ve IQR aykırı değerden neredeyse etkilenmez (SAĞLAM)')
print('→ Ortalama ve standart sapma dramatik biçimde değişir (HASSas)')

# Görselleştirme
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
titles = ['Orijinal', 'En Büyük → 10M', 'En Küçük → 10M']
datasets = [gelir, gelir_buyuk_degisti, gelir_kucuk_degisti]

for ax, data, title in zip(axes, datasets, titles):
    plot_data = data[data < 2_000_000]  # görsellik için sınırla
    ax.hist(plot_data, bins=25, color=BLUE, alpha=0.75, edgecolor='white')
    ax.axvline(np.median(data), color=GREEN,  linewidth=2, linestyle='-',  label=f'Med={np.median(data)/1e3:.0f}K')
    ax.axvline(np.mean(data),   color=ORANGE, linewidth=2, linestyle='--', label=f'Ort={np.mean(data)/1e3:.0f}K')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Yıllık Hane Geliri (₺)', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Sağlam İstatistikler: Aykırı Değerin Etkisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 Yan Yana Kutu Grafikleri — Gruplar Arası Karşılaştırma

In [ ]:
# Sınıf yılına göre kulüp üyeliği sayısı
np.random.seed(10)
siniflar = {
    '1. Sınıf':    np.random.poisson(2.5, 80),
    'Sophomore':   np.random.poisson(3.0, 80),
    '3. Sınıf':    np.random.poisson(3.2, 80),
    '4. Sınıf':    np.random.poisson(2.8, 80),
}

fig, ax = plt.subplots(figsize=(9, 5))
data_list  = list(siniflar.values())
labels     = list(siniflar.keys())

bp = ax.boxplot(data_list, labels=labels, patch_artist=True,
                boxprops=dict(facecolor=LBLUE, color=BLUE, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2.5),
                whiskerprops=dict(color=BLUE),
                flierprops=dict(marker='o', color=ORANGE, alpha=0.6, markersize=6))

# Ortalama noktaları
for i, data in enumerate(data_list, 1):
    ax.plot(i, np.mean(data), marker='D', color=ORANGE, markersize=8, zorder=5,
            label='Ortalama' if i == 1 else '')

ax.set_xlabel('Sınıf Yılı', fontsize=11)
ax.set_ylabel('Üye Olunan Kulüp Sayısı', fontsize=11)
ax.set_title('Sınıf Yılına Göre Kulüp Üyeliği Dağılımı\n(Yan Yana Kutu Grafiği)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Grup Özetleri:')
for label, data in siniflar.items():
    print(f'  {label:12s}: Ort={np.mean(data):.2f}  Med={np.median(data):.1f}  IQR={np.percentile(data,75)-np.percentile(data,25):.1f}')

---
<a id='3'></a>
## 3. Logaritmik Dönüşüm

> Veriler **aşırı çarpık** olduğunda logaritmik dönüşüm modellemeyi kolaylaştırır:
> - Aykırı değerler daha az belirgin hale gelir
> - Dağılım daha simetrik olur
>
> **Dezavantaj:** Logaritma birimindeki sonuçları yorumlamak güçleşir.

In [ ]:
# Basketbol maçı katılım verisi (sağa çarpık)
np.random.seed(5)
mac_sayisi = np.random.exponential(8, 200)
mac_sayisi = np.clip(mac_sayisi, 0.5, 75)  # en az 0.5 — log(0) tanımsız

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Ham veri
ax1 = axes[0]
ax1.hist(mac_sayisi, bins=25, color=BLUE, alpha=0.75, edgecolor='white')
ax1.axvline(np.mean(mac_sayisi),   color=ORANGE, lw=2, linestyle='--', label=f'Ort={np.mean(mac_sayisi):.1f}')
ax1.axvline(np.median(mac_sayisi), color=GREEN,  lw=2, linestyle='-',  label=f'Med={np.median(mac_sayisi):.1f}')
ax1.set_xlabel('Katılınan Maç Sayısı', fontsize=11)
ax1.set_ylabel('Frekans', fontsize=11)
ax1.set_title(f'Ham Veri  (çarpıklık = {stats.skew(mac_sayisi):.2f})', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)

# Log dönüşüm
log_mac = np.log(mac_sayisi)
ax2 = axes[1]
ax2.hist(log_mac, bins=25, color=GREEN, alpha=0.75, edgecolor='white')
ax2.axvline(np.mean(log_mac),   color=ORANGE, lw=2, linestyle='--', label=f'Ort={np.mean(log_mac):.2f}')
ax2.axvline(np.median(log_mac), color='navy', lw=2, linestyle='-',  label=f'Med={np.median(log_mac):.2f}')
ax2.set_xlabel('log(Katılınan Maç Sayısı)', fontsize=11)
ax2.set_ylabel('Frekans', fontsize=11)
ax2.set_title(f'Log Dönüşüm  (çarpıklık = {stats.skew(log_mac):.2f})', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Logaritmik Dönüşüm: Çarpıklığı Azaltma', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Karşılaştırma tablosu
ornek_degisim = pd.DataFrame({
    'Maç Sayısı': [70, 50, 25, 10, 1],
    'log(Maç Sayısı)': [round(np.log(x), 2) for x in [70, 50, 25, 10, 1]]
})
print('Dönüşüm Tablosu:')
print(ornek_degisim.to_string(index=False))

---
<a id='4'></a>
## 4. Kategorik Verilerin İncelenmesi

### 4.1 Çapraz Tablolar

> **Çapraz tablo (contingency table):** İki kategorik değişkeni birlikte özetleyen tablo.  
> Sunumdaki **Titanic** verisi esas alınmıştır.

In [ ]:
# Titanic verisini yeniden oluştur
titanic_df = pd.DataFrame({
    'Yaş':      ['Yetişkin'] * 2092 + ['Çocuk'] * 109,
    'Hayatta':  (['Öldü'] * 1438 + ['Kurtuldu'] * 654) +
                (['Öldü'] * 52   + ['Kurtuldu'] * 57)
})

# Çapraz tablo — frekans
cross_freq = pd.crosstab(
    titanic_df['Yaş'], titanic_df['Hayatta'],
    margins=True, margins_name='Toplam'
)
print('Çapraz Tablo — Frekanslar')
print(cross_freq)

# Çapraz tablo — satır oranları (koşullu olasılık)
cross_row = pd.crosstab(
    titanic_df['Yaş'], titanic_df['Hayatta'],
    normalize='index'
).round(3)
print('\nÇapraz Tablo — Satır Oranları (Koşullu Olasılık)')
print(cross_row)

pct_yetiskin = 654 / 2092 * 100
pct_cocuk    = 57  / 109  * 100
print(f'\nYetişkinlerin hayatta kalma oranı: {pct_yetiskin:.1f}%')
print(f'Çocukların hayatta kalma oranı:   {pct_cocuk:.1f}%')
print(f'\n→ Çocuklar yetişkinlere kıyasla {pct_cocuk - pct_yetiskin:.1f} puan daha fazla oranda kurtulmuştur.')

### 4.2 Çubuk Grafikler

Sunumdaki üç görselleştirme türü: yığılmış, yan yana, standartlaştırılmış.

In [ ]:
# Veriler
kategoriler  = ['Yetişkin', 'Çocuk']
oldu         = [1438, 52]
kurtuldu     = [654, 57]
toplamlar    = [o + k for o, k in zip(oldu, kurtuldu)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x = np.arange(len(kategoriler))
w = 0.5

# 1) Yığılmış çubuk grafik
ax = axes[0]
ax.bar(x, oldu,     width=w, label='Öldü',     color=BLUE,  alpha=0.85)
ax.bar(x, kurtuldu, width=w, label='Kurtuldu', color=LBLUE, alpha=0.85, bottom=oldu)
ax.set_xticks(x); ax.set_xticklabels(kategoriler, fontsize=11)
ax.set_ylabel('Frekans'); ax.set_title('Yığılmış Çubuk Grafik', fontweight='bold')
ax.legend(fontsize=9)

# 2) Yan yana çubuk grafik
ax2 = axes[1]
ax2.bar(x - w/4, oldu,     width=w/2, label='Öldü',     color=BLUE,  alpha=0.85)
ax2.bar(x + w/4, kurtuldu, width=w/2, label='Kurtuldu', color=LBLUE, alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(kategoriler, fontsize=11)
ax2.set_ylabel('Frekans'); ax2.set_title('Yan Yana Çubuk Grafik', fontweight='bold')
ax2.legend(fontsize=9)

# 3) Standartlaştırılmış (göreli frekans)
ax3 = axes[2]
oldu_oran     = [o / t for o, t in zip(oldu,     toplamlar)]
kurtuldu_oran = [k / t for k, t in zip(kurtuldu, toplamlar)]
ax3.bar(x, oldu_oran,     width=w, label='Öldü',     color=BLUE,  alpha=0.85)
ax3.bar(x, kurtuldu_oran, width=w, label='Kurtuldu', color=LBLUE, alpha=0.85, bottom=oldu_oran)
ax3.axhline(0.5, color='white', linewidth=1.5, linestyle='--')
ax3.set_xticks(x); ax3.set_xticklabels(kategoriler, fontsize=11)
ax3.set_ylabel('Göreli Frekans (Oran)'); ax3.set_title('Standartlaştırılmış Yığılmış\nÇubuk Grafik', fontweight='bold')
ax3.legend(fontsize=9)
ax3.set_ylim(0, 1)

plt.suptitle('Titanic — Yaş ve Hayatta Kalma: Üç Görselleştirme', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Mozaik Grafiği

> **Mozaik grafiği**, standartlaştırılmış yığılmış çubuk grafikten farklı olarak sütun **genişliklerini de** veri büyüklüğüne göre ölçekler.

In [ ]:
from matplotlib.patches import Rectangle

toplam_n = 2092 + 109  # 2201

# Genişlikler: her grubun toplam içindeki payı
genislik_yetiskin = 2092 / toplam_n
genislik_cocuk    = 109  / toplam_n

# Yükseklikler: her grup içindeki oranlar
oran_oldu_yetiskin     = 1438 / 2092
oran_kurtuldu_yetiskin = 654  / 2092
oran_oldu_cocuk        = 52   / 109
oran_kurtuldu_cocuk    = 57   / 109

fig, ax = plt.subplots(figsize=(9, 6))
gap = 0.02
x_yetiskin = 0
x_cocuk    = genislik_yetiskin + gap

# Yetişkin — Öldü
ax.add_patch(Rectangle((x_yetiskin, oran_kurtuldu_yetiskin),
                         genislik_yetiskin, oran_oldu_yetiskin,
                         color=BLUE, alpha=0.85))
# Yetişkin — Kurtuldu
ax.add_patch(Rectangle((x_yetiskin, 0),
                         genislik_yetiskin, oran_kurtuldu_yetiskin,
                         color=LBLUE, alpha=0.85))
# Çocuk — Öldü
ax.add_patch(Rectangle((x_cocuk, oran_kurtuldu_cocuk),
                         genislik_cocuk, oran_oldu_cocuk,
                         color=BLUE, alpha=0.85))
# Çocuk — Kurtuldu
ax.add_patch(Rectangle((x_cocuk, 0),
                         genislik_cocuk, oran_kurtuldu_cocuk,
                         color=LBLUE, alpha=0.85))

# Sınır çizgileri
for xpos, genislik, oran_k in [
    (x_yetiskin, genislik_yetiskin, oran_kurtuldu_yetiskin),
    (x_cocuk,    genislik_cocuk,    oran_kurtuldu_cocuk)
]:
    ax.plot([xpos, xpos + genislik], [oran_k, oran_k], 'white', linewidth=2.5)

# Etiketler
ax.text(x_yetiskin + genislik_yetiskin/2, 0.5,  'Yetişkin', ha='center', va='center',
        fontsize=13, fontweight='bold', color='white')
ax.text(x_cocuk    + genislik_cocuk/2,    0.5,  'Çocuk',    ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')
ax.text(-0.06, oran_kurtuldu_yetiskin + oran_oldu_yetiskin/2, 'Öldü',    ha='right', fontsize=11)
ax.text(-0.06, oran_kurtuldu_yetiskin/2,                      'Kurtuldu',ha='right', fontsize=11)

# Boyut bilgisi
ax.text(x_yetiskin + genislik_yetiskin/2, -0.07,
        f'n = 2092\n({genislik_yetiskin*100:.1f}%)', ha='center', fontsize=9, color=GRAY)
ax.text(x_cocuk + genislik_cocuk/2, -0.07,
        f'n = 109\n({genislik_cocuk*100:.1f}%)', ha='center', fontsize=9, color=GRAY)

legend_patches = [mpatches.Patch(color=BLUE,  label='Öldü'),
                  mpatches.Patch(color=LBLUE, label='Kurtuldu')]
ax.legend(handles=legend_patches, loc='upper right', fontsize=10)
ax.set_xlim(-0.08, 1.0)
ax.set_ylim(-0.15, 1.05)
ax.set_xlabel('Yaş Grubu  (sütun genişliği = grup büyüklüğü)', fontsize=10)
ax.set_ylabel('Oran', fontsize=10)
ax.set_title('Mozaik Grafiği — Titanic (Yaş × Hayatta Kalma)', fontsize=12, fontweight='bold')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xticks([])
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Örnek Çalışma: Cinsiyete Dayalı Ayrımcılık & Hipotez Testi

**Çalışma:** 48 erkek banka yöneticisine aynı personel dosyası verilmiştir.  
- 24 yönetici dosyayı **erkek** adayın, 24'ü **kadın** adayın dosyası olarak görmüştür.
- 35 dosya terfi için önerilmiştir.

**Hipotezler:**  
- $H_0$: Terfi ile cinsiyet **bağımsızdır** — fark yalnızca şansa bağlıdır.  
- $H_A$: Terfi ile cinsiyet **bağımlıdır** — cinsiyete dayalı ayrımcılık vardır.

In [ ]:
# Gözlemlenen veri
goz_erkek_terfi   = 21
goz_kadin_terfi   = 14
n_erkek = n_kadin = 24

goz_fark = goz_erkek_terfi/n_erkek - goz_kadin_terfi/n_kadin

df_gozlem = pd.DataFrame({
    'Terfi Etti':    [21, 14, 35],
    'Terfi Etmedi':  [ 3, 10, 13],
    'Toplam':        [24, 24, 48]
}, index=['Erkek', 'Kadın', 'Toplam'])

print('Gözlemlenen Veri:')
print(df_gozlem)
print(f'\nTerfi eden erkek oranı : {goz_erkek_terfi}/{n_erkek} = {goz_erkek_terfi/n_erkek:.3f} (%{goz_erkek_terfi/n_erkek*100:.1f})')
print(f'Terfi eden kadın oranı : {goz_kadin_terfi}/{n_kadin} = {goz_kadin_terfi/n_kadin:.3f} (%{goz_kadin_terfi/n_kadin*100:.1f})')
print(f'Gözlemlenen fark       : {goz_fark:.3f} (%{goz_fark*100:.1f})')

In [ ]:
# Permütasyon testi (Sunumdaki iskambil kart simülasyonunun yazılım versiyonu)
np.random.seed(2024)

n_sim  = 10_000
n_terfi = 35  # toplam terfi sayısı sabit
n_toplam = 48

# Her simülasyonda: 48 dosyayı karıştır, ilk 24'ü erkek, son 24'ü kadın say
dosyalar = np.array([1]*35 + [0]*13)  # 1=terfi etti, 0=etmedi

sim_farklar = []
for _ in range(n_sim):
    np.random.shuffle(dosyalar)
    erkek_oran = dosyalar[:24].mean()
    kadin_oran = dosyalar[24:].mean()
    sim_farklar.append(erkek_oran - kadin_oran)

sim_farklar = np.array(sim_farklar)

# p-değeri: gözlemlenen fark kadar uç değer gelme olasılığı
p_deger = np.mean(sim_farklar >= goz_fark)

# Görselleştirme
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(sim_farklar, bins=40, color=BLUE, alpha=0.7, edgecolor='white',
        label=f'Simülasyon dağılımı (n={n_sim:,})')
ax.axvline(goz_fark, color=ORANGE, linewidth=3,
           label=f'Gözlemlenen fark = {goz_fark:.3f}')

# Gözlemlenen farktan büyük veya eşit olan simülasyonlar
ax.hist(sim_farklar[sim_farklar >= goz_fark], bins=40,
        color=ORANGE, alpha=0.6, edgecolor='white', label=f'p-değeri bölgesi')

ax.set_xlabel('Terfi Oranlarındaki Fark (Erkek − Kadın)', fontsize=11)
ax.set_ylabel('Frekans', fontsize=11)
ax.set_title('Permütasyon Testi: Terfi Oranı Farkının Null Dağılımı\n'
             '(H₀: Cinsiyet ve terfi bağımsızdır)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# p-değeri kutusu
ax.text(0.97, 0.92, f'p-değeri = {p_deger:.4f}',
        transform=ax.transAxes, ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print('=' * 55)
print(f'Simülasyon sayısı          : {n_sim:,}')
print(f'Gözlemlenen fark           : {goz_fark:.4f}  (%{goz_fark*100:.1f})')
print(f'Simüle edilmiş p-değeri    : {p_deger:.4f}')
print('=' * 55)
if p_deger < 0.05:
    print('KARAR (α=0.05): H₀ REDDEDİLİR.')
    print('→ Cinsiyete dayalı ayrımcılık için istatistiksel kanıt vardır.')
else:
    print('KARAR (α=0.05): H₀ reddedilemez.')
    print('→ Gözlemlenen fark şansa bağlı olabilir.')

In [ ]:
# Ek: Ki-kare testi (teorik yöntem)
from scipy.stats import chi2_contingency

contingency = np.array([[21, 3],   # erkek: terfi etti, etmedi
                         [14, 10]]) # kadın: terfi etti, etmedi

chi2, p_chi2, dof, expected = chi2_contingency(contingency, correction=False)

print('Ki-Kare Bağımsızlık Testi')
print(f'χ² istatistiği = {chi2:.4f}')
print(f'Serbestlik derecesi = {dof}')
print(f'p-değeri = {p_chi2:.4f}')
print(f'\nBeklenen frekanslar:')
print(pd.DataFrame(expected.round(2),
                   index=['Erkek','Kadın'],
                   columns=['Terfi Etti','Terfi Etmedi']))
if p_chi2 < 0.05:
    print('\n→ p < 0.05: H₀ reddedilir. Cinsiyet ve terfi bağımlıdır.')

---
## Özet: Hipotez Testi Çerçevesi

| Adım | Açıklama |
|------|----------|
| 1 | **H₀** (sıfır hipotez): Statükoya göre "hiçbir şey olmadığı" iddiası |
| 2 | **Hₐ** (alternatif hipotez): Araştırma sorusu — "bir şeyler oluyor" |
| 3 | H₀ **doğruymuş gibi** test istatistiğinin dağılımını simüle et |
| 4 | **p-değeri** hesapla: Gözlemlenen sonucun en az bu kadar uç olma olasılığı |
| 5 | p < α ise H₀'ı **reddet**; aksi hâlde H₀'da **kal** (asla "H₀ doğrudur" deme!) |
